In [1]:
# # # Clean out duplicate drivers that cause factory registration errors
# # !pip uninstall -y jax-cuda12-plugin jax-cuda13-plugin jaxlib
# # !pip install -U "jax[cuda12]==0.9.1"

# # 1. Clean up everything first to avoid partial installation issues
# !pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda13-plugin

# # 2. Reinstall with explicit versioning and upgrade flags
# !pip install --upgrade jax==0.9.1 jaxlib==0.9.1 jax-cuda12-plugin==0.9.1

# Position-Conditioned Noise Encoder based Quantum GAN

In [2]:
# !pip install jax jaxlib

In [3]:
# !pip install -U "jax[cuda13]"

In [4]:
# # 1. Nuclear option: remove all potential JAX conflicts
# !pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda13-plugin jax-cuda12-pjrt jax-cuda13-pjrt

# # 2. Reinstall ONLY the one that matches your drivers (usually cuda12 is safest)
# !pip install -U "jax[cuda12]"

In [7]:
# !pip install pennylane

In [8]:
# # Verify with the new 2026 JAX backend check
# import jax
# from jax.extend import backend
# try:
#     print(f"Active Backend: {backend.get_backend().platform}")
#     print(f"Devices: {jax.devices()}")
# except Exception as e:
#     print(f"Error: {e}")

import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torch.utils.data

import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torchvision import datasets, transforms
import torchvision.transforms as transforms

from torch.utils.dlpack import from_dlpack as f2d, to_dlpack as t2d
# from jax.dlpack import from_dlpack as d2j, to_dlpack as j2d
from jax.dlpack import from_dlpack as d2j
# from jax.numpy import from_dlpack as d2j, to_dlpack as j2d
# from jax.numpy import from_dlpack as d2j

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import pennylane as qml

from tqdm import tqdm
from matplotlib import cm
from scipy.linalg import sqrtm
from sklearn.decomposition import PCA

import tensorcircuit as tc

K = tc.set_backend("jax")
print("K:", K)

import time

ModuleNotFoundError: No module named 'gast'

In [ ]:
import os
import random

import jax
import jax.numpy as jnp

os.environ['TORCH_USE_CUDA_DSA'] = "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_FLAGS"] = "--xla_gpu_force_compilation_parallelism=1"

# Verify with the new 2026 JAX backend check
import jax
from jax.extend import backend
try:
    print(f"Active Backend: {backend.get_backend().platform}")
    print(f"Devices: {jax.devices()}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
cudnn.benchmark = True

if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [2]:
def get_train_data(dataset='MNIST', ds_class=5):
    if dataset == 'MNIST':
        train_loader = torch.utils.data.DataLoader(datasets.MNIST('../mnist', 
                                                                download=True, 
                                                                train=True,
                                                                
                                                                transform=transforms.Compose([
                                                                    torchvision.transforms.ToTensor(),
                                                                    transforms.Lambda(torch.flatten),
                                                                ])), 
                                                batch_size=10000, 
                                                shuffle=True)
    elif dataset == 'Fashion':
        train_loader = torch.utils.data.DataLoader(datasets.FashionMNIST('../fashion', 
                                                                download=True, 
                                                                train=True,
                                                                
                                                                transform=transforms.Compose([
                                                                    torchvision.transforms.ToTensor(),
                                                                    transforms.Lambda(torch.flatten),
                                                                ])), 
                                                batch_size=10000, 
                                                shuffle=True)
    train_data = []
    label_to_keep = ds_class
    label_to_keep_name = str(label_to_keep)
    for (data, labels) in train_loader:
        for x, y in zip(data, labels):
            if y == label_to_keep:
                train_data.append(x.numpy())

    return label_to_keep_name, train_data

In [3]:
def scale_data(data, scale=None, dtype=np.float32):
    if scale is None:
        scale = [-1, 1]
    min_data, max_data = [float(np.min(data)), float(np.max(data))]
    min_scale, max_scale = [float(scale[0]), float(scale[1])]
    data = ((max_scale - min_scale) * (data - min_data) / (max_data - min_data)) + min_scale
    return data.astype(dtype)

In [4]:
# Function from https://machinelearningmastery.com/how-to-implement-the-frechet-inception-distance-fid-from-scratch/
def calculate_fid(act1, act2):
    mu1, sigma1 = act1.mean(axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = act2.mean(axis=0), np.cov(act2, rowvar=False)
    ssdiff = np.sum((mu1 - mu2)**2.0)
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return fid

In [12]:
label_to_keep_name, train_data = get_train_data(dataset='MNIST')

In [13]:
train_data = scale_data(np.array(train_data), [0,1])

In [34]:
pca = PCA(n_components=pca_dims)
pca_data_full = pca.fit_transform(train_data)
ordering = []

In [36]:
label_to_keep_name

'5'

In [37]:
pca_data_full.shape, train_data.shape

((5421, 40), (5421, 784))

In [38]:
for i in range(8):
    k = 4*i
    l = [i, 39-k, 38-k, 37-k, 36-k]
    ordering.append(l)
pca_min, pca_max = np.min(pca_data_full), np.max(pca_data_full)

In [83]:
image_size = 5  
position_encoding = 1
batch_size = 32
pca_dims=40
n_qubits = 5 + (position_encoding)
q_depth = 6 
n_generators = 8

In [84]:
full_train_data = [(i,j) for i,j in zip(scale_data(pca_data_full), train_data)]

transform = transforms.Compose([transforms.ToTensor()])
dataloader = torch.utils.data.DataLoader(
    scale_data(pca_data_full), batch_size=batch_size, shuffle=True, drop_last=True
)

In [85]:
class D_PCNE_QGAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(pca_dims, 64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)

In [110]:
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
# @qml.qnode(dev, interface="jax", diff_method="parameter-shift")
def quantum_circuit_PCNE_QGAN(inputs, weights, weights_ne):  # input: (32, 8, 5) -> (32, 8, 5+1)
    # # weights = weights.reshape(q_depth, n_qubits)
    # for i in range(n_qubits):
    #     qml.RY(inputs[i], wires=i)
    #     qml.RX(inputs[i], wires=i)
    qml.AngleEmbedding(features=inputs, wires=range(n_qubits), rotation='Y')
    qml.AngleEmbedding(features=inputs, wires=range(n_qubits), rotation='X')

    q_depth_ne, n_qubits_full = weights_ne.shape
    q_depth_vqc, n_qubits_vqc = weights.shape

    assert n_qubits_full == n_qubits_vqc + position_encoding
    assert n_qubits_full == n_qubits
    
    for i in range(q_depth_ne):
        for y in range(n_qubits):
            qml.RY(weights_ne[i][y], wires=y)
        for y in range(n_qubits):
            qml.CZ(wires=[y, (y + 1)%n_qubits])

    for i in range(q_depth_vqc):
        for y in range(n_qubits_vqc):
            qml.RY(weights[i][y], wires=y)
        for y in range(n_qubits_vqc):
            qml.CZ(wires=[y, (y + 1)%n_qubits_vqc])
    
    return [qml.expval(qml.PauliX(i)) for i in range(n_qubits_vqc)]

In [116]:
ordering

[[0, 39, 38, 37, 36],
 [1, 35, 34, 33, 32],
 [2, 31, 30, 29, 28],
 [3, 27, 26, 25, 24],
 [4, 23, 22, 21, 20],
 [5, 19, 18, 17, 16],
 [6, 15, 14, 13, 12],
 [7, 11, 10, 9, 8]]

In [140]:
jax_circuit__1 = jax.jit(quantum_circuit_PCNE_QGAN)

# vmap_circuit = jax.vmap(jax_circuit__1, in_axes=(0, None, None))
vmap_circuit = jax.vmap(jax_circuit__1, in_axes=(0, None, None))

In [141]:
class JAXQuantumFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, inputs, weights, weights_ne):
        jax_gpu = jax.devices("gpu")[0]
        jax_in = jax.device_put(d2j(t2d(inputs.detach())), jax_gpu)
        jax_w = jax.device_put(d2j(t2d(weights.detach())), jax_gpu)
        jax_w_ne = jax.device_put(d2j(t2d(weights_ne.detach())), jax_gpu)
        # jax_mixing_weights = jax.device_put(d2j(t2d(mixing_weights.detach())), jax_gpu)
        
        # 1. Define a wrapper that ensures the output is ALWAYS a JAX array
        # This allows JAX to 'trace' the stacking logic
        def consolidated_model(i, w, w_ne):
            out = vmap_circuit(i, w, w_ne)
            if isinstance(out, (list, tuple)):
                return jnp.stack(out, axis=-1)
            return out

        # 2. Compute VJP on the consolidated model
        val, jax_vjp_fn = jax.vjp(consolidated_model, jax_in, jax_w, jax_w_ne)
        
        ctx.jax_vjp_fn = jax_vjp_fn
        # Return to Torch
        return f2d(j2d(val))

    @staticmethod
    def backward(ctx, grad_output):
        # 1. Detect device from the incoming gradient
        current_device = grad_output.device
        jax_gpu = jax.devices("gpu")[0]
        
        # 2. Convert to JAX for calculation
        jax_grad_out = jax.device_put(d2j(t2d(grad_output.detach())), jax_gpu)
        
        # 3. JAX backward pass
        # grad_inputs, grad_weights, grad_mixing = ctx.jax_vjp_fn(jax_grad_out)
        grad_inputs, grad_weights, grad_weights_ne = ctx.jax_vjp_fn(jax_grad_out)
        
        # 4. Convert back to Torch and FORCE the device metadata
        # We use .to(current_device) to ensure it matches exactly
        return (
            f2d(j2d(grad_inputs)).to(current_device), 
            f2d(j2d(grad_weights)).to(current_device),
            f2d(j2d(grad_weights_ne)).to(current_device)
            # f2d(j2d(grad_mixing)).to(current_device)
        )

In [163]:
class QuantumMLP(nn.Module):
    def __init__(self, q_delta, q_depth, q_depth_ne, n_qubits, position_encoding, name='QuantumMLP'):
        super().__init__()
        self.name = name
        # Define weights as PyTorch parameters so Adam can optimize them
        # self.vqc_weights = nn.Parameter(torch.randn(weights_shape).to(torch.device('cuda')))
        # self.vqc_weights = nn.Parameter(torch.zeros(weights_shape).to(torch.device('cuda')))
        
        self.q_params = nn.ParameterList(
            [nn.Parameter(q_delta * torch.rand(q_depth, n_qubits-position_encoding), requires_grad=True),
             nn.Parameter(q_delta * torch.rand(q_depth_ne, n_qubits), requires_grad=True)]
        )
        
        # self.mixing_weights = nn.Parameter(torch.randn(mixing_weights).to(torch.device('cuda')))

    def forward(self, x):
        out = JAXQuantumFunction.apply(x, self.q_params[0], self.q_params[1])
        
        return out

In [164]:
class QuantumGenerator_PCNE_QGAN(nn.Module):
    def __init__(self, n_generators, position_encoding, q_depth_ne, q_depth, q_delta=1, env='simulation'):
        super().__init__()

        # self.q_params = nn.ParameterList(
        #     [
        #         nn.Parameter(q_delta * torch.rand(q_depth, n_qubits), requires_grad=True)
        #         for _ in range(1)
        #     ]
        # )
        # self.q_params = [nn.Parameter(q_delta * torch.rand(q_depth, n_qubits-position_encoding), requires_grad=True),
        #                  nn.Parameter(q_delta * torch.rand(q_depth_ne, n_qubits), requires_grad=True)]

        self.n_generators = n_generators
        self.env = env

        # weight_shapes = {"weights": (q_depth, n_qubits)}
        # if env == "real":
        #     self.qcircuit = qml.qnn.TorchLayer(quantum_cirtui_real_machine, weight_shapes).to(device)
        # else:
        #     self.qcircuit = qml.qnn.TorchLayer(quantum_circuit_MosaiQ, weight_shapes).to(device)

        # weight_shapes = {
        #     "weights": (q_depth, n_qubits-position_encoding),
        #     "weights_ne": (q_depth_ne, n_qubits),
        # }
        
        # init_method = {
        #     "weights": nn.Parameter(q_delta * torch.rand(q_depth, n_qubits-position_encoding), requires_grad=True),
        #     "weights_ne":  nn.Parameter(q_delta * torch.rand(q_depth_ne, n_qubits), requires_grad=True),
        # }
        # self.generator_torch_layer = qml.qnn.TorchLayer(quantum_circuit_PCNE_QGAN, weight_shapes=weight_shapes, init_method=init_method).to(device)
        
        self.generator_torch_layer = QuantumMLP(q_delta, q_depth, q_depth_ne, n_qubits, position_encoding).to(device)

    def forward(self, x):
        # print("x shape:", x.shape)  # (batch_size, length of noise)
        images = []
        patch_size = image_size
        images = torch.Tensor(x.size(0), 0).to(device)
        batch_size, len_data = x.shape

        x = x.unsqueeze(1)  # (32, 1, 5)
        x = x.expand(-1, n_generators, -1)  # (32, 8, 5)
        # print("x:", x)
        pos_values = torch.linspace(np.pi/4, 2*np.pi, steps=n_generators).to(device)
        pos_values = pos_values.view(1, n_generators, 1)
        pos_values = pos_values.expand(batch_size, -1, -1)  # (32, 8, 5)
        pos_values = pos_values.view(batch_size, n_generators, 1)
        x = torch.cat((x, pos_values), dim=2)
        # print("x:", x)
        # print("x shape:", x.shape)

        output = self.generator_torch_layer(x)
        # output = JAXQuantumFunction.apply(x, self.q_params[0], self.q_params[1])
        # print("output shape:", output.shape)

        output = output.view(output.size(0), -1)
        flattened_order = [j for sub in ordering for j in sub]
        output = output[:, flattened_order]
        # print("output shape:", output.shape)
        # output = output.reshape(batch_size, patch_size)
        # print("patches shape:", output.shape)
        
        return output

In [165]:
# flattened_order = [j for sub in ordering for j in sub]
# flattened_order

In [166]:
position_encoding

1

In [167]:
lrG = 0.3
lrD = 0.05
num_iter = 1

gen_losses = []
disc_losses = []
discriminator = D_PCNE_QGAN().to(device)
generator = QuantumGenerator_PCNE_QGAN(n_generators, position_encoding, 3, 3).to(device)
criterion = nn.BCELoss()

In [168]:
for name, param in generator.named_parameters():
    if param.requires_grad:
        print(f"Layer name: {name} | Trainable: {param.requires_grad} | Size: {param.size()}")

Layer name: generator_torch_layer.q_params.0 | Trainable: True | Size: torch.Size([3, 5])
Layer name: generator_torch_layer.q_params.1 | Trainable: True | Size: torch.Size([3, 6])


In [169]:
optD = optim.SGD(discriminator.parameters(), lr=lrD)
optG = optim.SGD(generator.parameters(), lr=lrG)
real_labels = torch.full((batch_size,), 1.0, dtype=torch.float, device=device)
fake_labels = torch.full((batch_size,), 0.0, dtype=torch.float, device=device)
counter = 0

# noise_upper_bound = math.pi/8

In [170]:
print('Training...')
for e in tqdm(range(num_iter)):
    for i, train_pair in enumerate(dataloader):
        pca_data = train_pair
        data = pca_data.reshape(batch_size, pca_dims)
        real_data = data.to(device).to(torch.float32)
        noise = torch.rand(batch_size, n_qubits-position_encoding, device=device)

        st = time.time()
        fake_data = generator(noise)
        print(i, "time taken:", time.time() - st)
        
        discriminator.zero_grad()
        outD_real = discriminator(real_data).view(-1)
        outD_fake = discriminator(fake_data.detach()).view(-1)
        errD_real = criterion(outD_real, real_labels)
        errD_fake = criterion(outD_fake, fake_labels)
        errD_real.backward()
        errD_fake.backward()
        errG = criterion(outD_fake, real_labels)
        errD = errD_real + errD_fake
        
        # gen_losses.append(errG.detach().cpu().numpy())
        # disc_losses.append(errD.detach().cpu().numpy())
        
        gen_losses.append(errG.detach().cpu().numpy())
        disc_losses.append(errD.detach().cpu().numpy())

        optD.step()

        # Train the generator
        generator.zero_grad()
        outD_fake = discriminator(fake_data).view(-1)
        errG = criterion(outD_fake, real_labels)
        errG.backward()
        optG.step()
        # if original_ratio is None:
        #     original_ratio = errD.detach().cpu().numpy()/errG.detach().cpu().numpy()
        # noise_upper_bound = get_noise_upper_bound(errG, errD, original_ratio)
        # upper_bounds.append(noise_upper_bound)
        # np.save(f'upper_bounds_{label_to_keep_name}', upper_bounds)
        counter += 1      
        if counter % 20 == 0:  
            test_images = generator(noise).detach().cpu().numpy()
            test_images = pca.inverse_transform(test_images)
            fid = calculate_fid(test_images.reshape([batch_size, 784]), train_data)
            test_images = scale_data(test_images,[0,1])
            real_images = []
            np.save(f'gen_loss_{label_to_keep_name}', gen_losses)
            np.save(f'disc_loss_{label_to_keep_name}', disc_losses)
            from PIL import Image 
            im = np.reshape(test_images[0], [28, 28])
            new_im = np.zeros([28,28])
            for i in range(28):
                for j in range(28):
                    if im[i][j] > .5:
                        new_im[i][j] = 0.0
                    else:
                        new_im[i][j] = 1.0
            im = Image.fromarray(np.uint8(255-(new_im*255)))
            
            # im = im.save(os.path.join("../QGAN/gen_images_dist",f"{label_to_keep_name}_{counter}.png"))  # original
            # im = im.save(os.path.join("../QGAN/gen_images_dist simple rot",f"{label_to_keep_name}_{counter}.png"))  # naive rotation
            im = im.save(os.path.join("../QGAN/gen_images_dist same generator no noise encoder",f"{label_to_keep_name}_{counter}.png"))  # naive rotation
            
            torch.save(generator.state_dict(), f"generator_{label_to_keep_name}")
            torch.save(discriminator.state_dict(), f"disc_{label_to_keep_name}")

Training...


  0%|                                                                        | 0/1 [00:00<?, ?it/s]


NameError: name 'd2j' is not defined